In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files # Esta es específica para subir archivos

Pasos realizados para la limpieza:
- Filtrado de columnas irrelevantes (url, nombres, img)
- Salvaguardas textos (mantener descripciones y autorellenar con "No description")
- Eliminación de filas sin métricas de rendimiento (juegos sin datos clave)
- Estandarización de fechas (fecha en texto plano y añadir column de release_year)
- Conversión de rangos comerciales a valores numéricos (owners_midpoint para estimar rangos)
- Reestructuración del índice (forzar columna indice a ser la primera)

Recomendaciones de limpieza:
- Filtrar los gigantes de la industria (cs2, dota)-solo indies
- Codificacion de generos (Accion 0, Rol, 1 - [Cada genero una columna independiente 0-no es de ese genero, 1 - si es])
- Diferenciar gratuitos con columna indepeneidente (0-no es, 1- si es)
- Normalización de variables de impacto (Aplicar una transformación logarítmica (np.log1p) a las columnas de jugadores y dueños.)


In [ ]:
df = pd.read_csv('games_march2025_cleaned.csv')

In [ ]:
# 1. Ver las columnas y tipos de datos
print("--- COLUMNAS ---")
print(df.info())

# 2. Ver una muestra de los datos reales
print("\n--- PRIMERAS FILAS ---")
print(df.head().to_csv())

# 3. Ver cuántos valores faltantes hay (importante para limpiar)
print("\n--- VALORES NULOS ---")
print(df.isnull().sum())

--- COLUMNAS ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3382 entries, 0 to 3381
Data columns (total 47 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   appid                     3382 non-null   int64  
 1   name                      3382 non-null   object 
 2   release_date              3382 non-null   object 
 3   required_age              3382 non-null   int64  
 4   price                     3382 non-null   float64
 5   dlc_count                 3382 non-null   int64  
 6   detailed_description      3358 non-null   object 
 7   about_the_game            3355 non-null   object 
 8   short_description         3370 non-null   object 
 9   reviews                   1217 non-null   object 
 10  header_image              3382 non-null   object 
 11  website                   2622 non-null   object 
 12  support_url               2511 non-null   object 
 13  support_email             2203 non-null   obje

In [ ]:
# 1. Lista de columnas a eliminar
cols_to_drop = [
    'name', 'dlc_count', 'achievements', 'developers',
    'reviews', 'header_image', 'website', 'support_url',
    'support_email', 'metacritic_url', 'notes', 'packages',
    'screenshots', 'movies', 'score_rank', 'discount'
]

# Aplicamos la limpieza asegurándonos de que 'appid' NO esté en la lista negra
df_clean = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# 2. Gestión de descripciones y valores nulos
desc_cols = ['detailed_description', 'about_the_game', 'short_description']
for col in desc_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna("No description")

# Limpieza de filas críticas (mantenemos solo juegos con datos de comunidad)
df_clean = df_clean.dropna(subset=['peak_ccu', 'tags', 'num_reviews_total'])

# 3. Procesamiento de fechas y conversión de Owners a número
df_clean['release_date'] = pd.to_datetime(df_clean['release_date'], errors='coerce')
df_clean['release_year'] = df_clean['release_date'].dt.year

def parse_owners(owner_str):
    if pd.isna(owner_str) or '-' not in str(owner_str):
        return 0
    parts = str(owner_str).replace(',', '').replace(' ', '').split('-')
    try:
        low, high = map(int, parts)
        return (low + high) / 2
    except:
        return 0

df_clean['owners_midpoint'] = df_clean['estimated_owners'].apply(parse_owners)

# 4. Forzamos que 'appid' sea la primera columna y eliminamos cualquier rastro de 'name'
cols = ['appid'] + [c for c in df_clean.columns if c not in ['appid', 'name']]
df_clean = df_clean[cols]

print(f"Dataset listo. Filas: {len(df_clean)}")
print("-" * 30)
print("Columnas actuales:")
print(df_clean.columns.tolist())

Dataset listo. Filas: 3381
------------------------------
Columnas actuales:
['appid', 'release_date', 'required_age', 'price', 'detailed_description', 'about_the_game', 'short_description', 'windows', 'mac', 'linux', 'metacritic_score', 'recommendations', 'supported_languages', 'full_audio_languages', 'publishers', 'categories', 'genres', 'user_score', 'positive', 'negative', 'estimated_owners', 'average_playtime_forever', 'average_playtime_2weeks', 'median_playtime_forever', 'median_playtime_2weeks', 'peak_ccu', 'tags', 'pct_pos_total', 'num_reviews_total', 'pct_pos_recent', 'num_reviews_recent', 'release_year', 'owners_midpoint']
